# 02_b_premodel_steps: Marketing-Aware Pre-Model Pipeline

This notebook converts outputs from 01_b, 01_c, and 02_a into a model-ready pipeline with explicit strategy choices.

Objectives:
1. Build leakage-safe train/validation split.
2. Prepare features with business constraints from EDA findings.
3. Train baseline models with consistent preprocessing.
4. Compare model quality using classification + ranking metrics.
5. Select decision threshold aligned to campaign objective.
6. Export prepared datasets, feature lists, and model artifacts.

## Step 1) Imports and Data Loading

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    f1_score, precision_score, recall_score, roc_auc_score, average_precision_score,
    mean_squared_error
    )
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

pd.set_option('display.max_columns', 200)

PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
INPUT_PATH = DATA_DIR / 'features_added' / 'preprocessed_data_features_added.csv'

if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Expected file not found: {INPUT_PATH}')

df = pd.read_csv(INPUT_PATH)
print('Loaded:', INPUT_PATH)
print('Shape:', df.shape)
df.head()

Loaded: c:\Nam\ddm\Data_marketing_starbuck_rewards\data\features_added\preprocessed_data_features_added.csv
Shape: (50637, 50)


,person,offer_id,time_completed,time_received,time_viewed,was_viewed,completed_after_view,within_offer_window,offer_success,received_not_viewed,age,income,days_since_registration,age_was_118,income_missing_before_impute,gender_missing_before_fill,age_missing_before_impute,reward,difficulty,channel_email,channel_mobile,channel_social,channel_web,channel_count,reward_to_difficulty,reward_per_day,offer_type_bogo,offer_type_discount,offer_type_informational,transaction_count,total_spent,avg_transaction_value,duration,time_completed_was_imputed,time_viewed_was_imputed,age_group,income_group,membership_duration_months,has_web_channel,has_email_channel,has_mobile_channel,has_social_channel,spend_per_transaction,spend_per_membership_day,difficulty_per_day,net_value_score,gender_F,gender_M,gender_O,gender_U
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,576.0,576.0,360.0,0,False,True,0,1,33,72000.0,461,0,0,0,0,2,10,1,1,0,1,3,0.20,0.285714,0,1,0,8.0,127.60,15.950000,7,0,1,26-35,middle,15.14,1,1,1,0,15.950000,0.276790,1.428571,1.0,False,True,False,False
1,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,414.0,408.0,456.0,1,False,True,0,0,33,72000.0,461,0,0,0,0,5,5,1,1,1,1,4,1.00,1.000000,1,0,0,8.0,127.60,15.950000,5,0,0,26-35,middle,15.14,1,1,1,1,15.950000,0.276790,1.000000,4.5,False,True,False,False
2,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,528.0,504.0,540.0,1,False,True,0,0,33,72000.0,461,0,0,0,0,2,10,1,1,1,1,4,0.20,0.200000,0,1,0,8.0,127.60,15.950000,10,0,0,26-35,middle,15.14,1,1,1,1,15.950000,0.276790,1.000000,1.0,False,True,False,False
3,00116118485d4dfda04fdbaba9a87b5c,f19421c1d4aa40978ebb69ca19b0e20d,420.0,168.0,216.0,1,False,False,0,0,55,64000.0,92,1,1,1,1,5,5,1,1,1,1,4,1.00,1.000000,1,0,0,3.0,4.09,1.363333,5,1,0,46-55,middle,3.02,1,1,1,1,1.363333,0.044457,1.000000,4.5,False,False,False,True
4,0011e0d4e6b944f998e987f904e8c1e5,0b1e1539f2cc45b7b9fa7c272da2e1d7,576.0,408.0,432.0,1,True,True,1,0,40,57000.0,198,0,0,0,0,5,20,1,0,0,1,2,0.25,0.500000,0,1,0,5.0,79.46,15.892000,10,0,0,36-45,lower_middle,6.50,1,1,0,0,15.892000,0.401313,2.000000,3.0,False,False,True,False


## Step 2) Target Definition and Governance Rules

Rules from EDA and business strategy:
- Target: `offer_success` (fallback to compatible success columns if needed).
- Remove obvious identifiers from features.
- Remove leakage-prone columns tied directly to completion outcomes.

In [2]:
if 'offer_success' in df.columns:
    target_col = 'offer_success'
elif 'offer_completed' in df.columns:
    target_col = 'offer_completed'
elif 'offer_completed_event' in df.columns:
    target_col = 'offer_completed_event'
else:
    raise ValueError('No target found: expected offer_success / offer_completed / offer_completed_event.')

df[target_col] = pd.to_numeric(df[target_col], errors='coerce').fillna(0).astype(int)

id_like_cols = [c for c in ['person', 'offer_id', 'offer'] if c in df.columns]
leakage_cols = [c for c in ['completed_after_view', 'within_offer_window', 'time_completed', 'time_completed_was_imputed'] if c in df.columns]

drop_feature_cols = set(id_like_cols + leakage_cols + [target_col])
feature_cols = [c for c in df.columns if c not in drop_feature_cols]

X = df[feature_cols].copy()
y = df[target_col].copy()

print('Target:', target_col)
print('Target balance (%):')
print((y.value_counts(normalize=True).sort_index() * 100).round(2))
print('ID-like dropped:', id_like_cols)
print('Leakage dropped:', leakage_cols)
print('Feature count:', len(feature_cols))

Target: offer_success
Target balance (%):
offer_success
0    61.02
1    38.98
Name: proportion, dtype: float64
ID-like dropped: ['person', 'offer_id']
Leakage dropped: ['completed_after_view', 'within_offer_window', 'time_completed', 'time_completed_was_imputed']
Feature count: 43


## Step 3) Group-Aware Train/Validation Split

To avoid customer leakage across train and validation, split by customer group (`person`) when available.

In [3]:
if 'person' in df.columns:
    groups = df['person'].astype(str)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, valid_idx = next(gss.split(X, y, groups=groups))
else:
    rng = np.random.default_rng(42)
    idx = np.arange(len(df))
    rng.shuffle(idx)
    cut = int(0.8 * len(idx))
    train_idx, valid_idx = idx[:cut], idx[cut:]

X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
y_train, y_valid = y.iloc[train_idx].copy(), y.iloc[valid_idx].copy()

print('Train shape:', X_train.shape, '| Valid shape:', X_valid.shape)
print('Train target rate:', round(y_train.mean(), 4))
print('Valid target rate:', round(y_valid.mean(), 4))

if 'person' in df.columns:
    overlap = set(df.iloc[train_idx]['person']).intersection(set(df.iloc[valid_idx]['person']))
    print('Customer overlap between train/valid:', len(overlap))

Train shape: (40510, 43) | Valid shape: (10127, 43)
Train target rate: 0.3897
Valid target rate: 0.3904
Customer overlap between train/valid: 0


## Step 4) Preprocessing Pipeline

- Numeric features: median imputation + standard scaling.
- Categorical features: most-frequent imputation + one-hot encoding.
- Build one reusable transformer for all baseline models.

In [4]:
# Convert boolean cols → int để chúng được xử lý như numeric
bool_cols = X_train.select_dtypes(include=['bool']).columns.tolist()
X_train[bool_cols] = X_train[bool_cols].astype(int)
X_valid[bool_cols] = X_valid[bool_cols].astype(int)

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)
], remainder='drop')

print('Numeric cols:', len(numeric_cols))
print('Categorical cols:', len(categorical_cols))

Numeric cols: 41
Categorical cols: 2


## Step 4b) Most Popular Offer Baseline (Naive Benchmark)

Không dùng model. Predict = tỷ lệ success trung bình theo từng offer trên train set.
Dùng để chứng minh supervised models có giá trị gia tăng.


In [5]:
# ── Most Popular Offer Baseline ─────────────────────────────────
train_offer_ids = df.iloc[train_idx]['offer_id']
valid_offer_ids = df.iloc[valid_idx]['offer_id']

# Completion rate trung bình theo offer trên train
offer_pop_rate = (
    pd.DataFrame({'offer_id': train_offer_ids, 'y': y_train})
    .groupby('offer_id')['y'].mean()
)

# Map sang valid set
valid_proba_popular = valid_offer_ids.map(offer_pop_rate).fillna(y_train.mean()).values
valid_pred_popular  = (valid_proba_popular >= 0.5).astype(int)

popular_metrics = {
    'model': 'most_popular_offer',
    'accuracy': accuracy_score(y_valid, valid_pred_popular),
    'balanced_accuracy': balanced_accuracy_score(y_valid, valid_pred_popular),
    'f1': f1_score(y_valid, valid_pred_popular),
    'precision': precision_score(y_valid, valid_pred_popular, zero_division=0),
    'recall': recall_score(y_valid, valid_pred_popular, zero_division=0),
    'roc_auc': roc_auc_score(y_valid, valid_proba_popular),
    'pr_auc': average_precision_score(y_valid, valid_proba_popular),
}

print("Most Popular Offer Baseline:")
for k, v in popular_metrics.items():
    if k != 'model':
        print(f"  {k}: {v:.4f}")

print("\nOffer-level popularity (train):")
print(offer_pop_rate.sort_values(ascending=False).to_string())


Most Popular Offer Baseline:
  accuracy: 0.6546
  balanced_accuracy: 0.6045
  f1: 0.4594
  precision: 0.5906
  recall: 0.3758
  roc_auc: 0.6695
  pr_auc: 0.5288

Offer-level popularity (train):
offer_id
fafdcd668e3743c1bb461111dcafc2a4    0.624754
2298d6c36e964ae4a3e7e9706d1fb8c2    0.583975
f19421c1d4aa40978ebb69ca19b0e20d    0.461185
4d5c57ea9a6940dd891ad53e9dbe8da0    0.356874
ae264e3637204a6fb9bb56bc8210ddfd    0.341587
9b98b8c7a33c4b65b9aebfe6a799e6d9    0.287295
2906b810c7d4411798c6938adc9daaa5    0.284459
0b1e1539f2cc45b7b9fa7c272da2e1d7    0.178578


## Step 5) Baseline Model Training and Evaluation

We train three strong baselines and compare both classification and ranking quality:
- Logistic Regression
- Random Forest
- Histogram Gradient Boosting

In [6]:
def fit_and_evaluate(model_name, estimator):
    pipe = Pipeline([
        ('prep', preprocessor),
        ('model', estimator)
    ])
    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_valid)
    if hasattr(pipe, 'predict_proba'):
        proba = pipe.predict_proba(X_valid)[:, 1]
    else:
        decision = pipe.decision_function(X_valid)
        proba = (decision - decision.min()) / (decision.max() - decision.min() + 1e-9)

    metrics = {
        'model': model_name,
        'accuracy': accuracy_score(y_valid, pred),
        'balanced_accuracy': balanced_accuracy_score(y_valid, pred),
        'f1': f1_score(y_valid, pred),
        'precision': precision_score(y_valid, pred, zero_division=0),
        'recall': recall_score(y_valid, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_valid, proba),
        'pr_auc': average_precision_score(y_valid, proba)
    }
    return pipe, metrics, pred, proba

models = {
    'log_reg': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'random_forest': RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=5,
        class_weight='balanced_subsample', random_state=42, n_jobs=-1
    ),
    'hist_gb': HistGradientBoostingClassifier(
        learning_rate=0.05, max_depth=8, max_iter=300, random_state=42
    )
}

results = []
fitted_models = {}
pred_cache = {}
proba_cache = {}

for name, est in models.items():
    pipe, m, pred, proba = fit_and_evaluate(name, est)
    fitted_models[name] = pipe
    pred_cache[name] = pred
    proba_cache[name] = proba
    results.append(m)

results_df = pd.DataFrame(results).sort_values('pr_auc', ascending=False).reset_index(drop=True)
results_df

,model,accuracy,balanced_accuracy,f1,precision,recall,roc_auc,pr_auc
0,hist_gb,0.846253,0.846794,0.811797,0.777495,0.849267,0.927503,0.872726
1,random_forest,0.824528,0.836702,0.798823,0.723099,0.892261,0.911085,0.832345
2,log_reg,0.807445,0.817962,0.778359,0.706854,0.865959,0.898018,0.807279


In [7]:
# ── MSE cho tất cả supervised models ────────────────────────────
print("\n=== MSE (predicted probability vs actual label) ===")
for name, proba in proba_cache.items():
    mse = mean_squared_error(y_valid, proba)
    print(f"  {name}: MSE = {mse:.4f}")



=== MSE (predicted probability vs actual label) ===
  log_reg: MSE = 0.1313
  random_forest: MSE = 0.1189
  hist_gb: MSE = 0.1062


## Step 5b) SVD Collaborative Filtering

Build user-offer interaction matrix và phân rã SVD.
So sánh với supervised baselines để đánh giá collaborative approach.


In [8]:
# ── SVD Collaborative Filtering ─────────────────────────────────
from scipy.sparse.linalg import svds

# 1. Build user-offer matrix (chỉ từ train data)
train_svd_df = df.iloc[train_idx][['person', 'offer_id', target_col]].copy()

user_offer_matrix = train_svd_df.pivot_table(
    index='person', columns='offer_id', values=target_col, fill_value=0
)
print(f"User-Offer Matrix: {user_offer_matrix.shape[0]} users × {user_offer_matrix.shape[1]} offers")
print(f"Sparsity: {(user_offer_matrix == 0).sum().sum() / user_offer_matrix.size:.1%}")

# 2. SVD decomposition (k latent factors)
k = min(5, user_offer_matrix.shape[1] - 1)
U, sigma, Vt = svds(user_offer_matrix.values.astype(float), k=k)
svd_pred_matrix = pd.DataFrame(
    (U @ np.diag(sigma) @ Vt).clip(0, 1),
    index=user_offer_matrix.index,
    columns=user_offer_matrix.columns,
)

# 3. Score validation set
valid_svd_df = df.iloc[valid_idx][['person', 'offer_id', target_col]].copy()
global_fallback = y_train.mean()

svd_scores = valid_svd_df.apply(
    lambda row: (
        svd_pred_matrix.loc[row['person'], row['offer_id']]
        if row['person'] in svd_pred_matrix.index
           and row['offer_id'] in svd_pred_matrix.columns
        else global_fallback
    ),
    axis=1,
)

valid_pred_svd = (svd_scores >= 0.5).astype(int)

# 4. Evaluate
svd_metrics = {
    'model': 'svd_collaborative',
    'accuracy': accuracy_score(y_valid, valid_pred_svd),
    'balanced_accuracy': balanced_accuracy_score(y_valid, valid_pred_svd),
    'f1': f1_score(y_valid, valid_pred_svd),
    'precision': precision_score(y_valid, valid_pred_svd, zero_division=0),
    'recall': recall_score(y_valid, valid_pred_svd, zero_division=0),
    'roc_auc': roc_auc_score(y_valid, svd_scores),
    'pr_auc': average_precision_score(y_valid, svd_scores),
}

svd_mse = mean_squared_error(y_valid, svd_scores)

print(f"\nSVD Collaborative (k={k}):")
for k_name, v in svd_metrics.items():
    if k_name != 'model':
        print(f"  {k_name}: {v:.4f}")
print(f"  mse: {svd_mse:.4f}")


User-Offer Matrix: 13542 users × 8 offers
Sparsity: 85.4%

SVD Collaborative (k=5):
  accuracy: 0.6096
  balanced_accuracy: 0.5000
  f1: 0.0000
  precision: 0.0000
  recall: 0.0000
  roc_auc: 0.5000
  pr_auc: 0.3904
  mse: 0.2380


## Step 6) Threshold Strategy for Campaign Activation

For campaign execution, threshold choice matters more than raw accuracy.
We sweep thresholds to balance precision and recall, then choose the best F1 threshold as a default operating point.

In [9]:
best_model_name = results_df.iloc[0]['model']
best_proba = proba_cache[best_model_name]

threshold_grid = np.linspace(0.10, 0.90, 33)
th_rows = []
for th in threshold_grid:
    p = (best_proba >= th).astype(int)
    th_rows.append({
        'threshold': round(float(th), 3),
        'precision': precision_score(y_valid, p, zero_division=0),
        'recall': recall_score(y_valid, p, zero_division=0),
        'f1': f1_score(y_valid, p, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_valid, p)
    })

th_df = pd.DataFrame(th_rows).sort_values('f1', ascending=False).reset_index(drop=True)
best_threshold = float(th_df.iloc[0]['threshold'])
best_pred = (best_proba >= best_threshold).astype(int)

print('Best model:', best_model_name)
print('Chosen threshold:', best_threshold)
print('Classification report at chosen threshold:')
print(classification_report(y_valid, best_pred, digits=4))
th_df.head(10)

Best model: hist_gb
Chosen threshold: 0.45
Classification report at chosen threshold:
              precision    recall  f1-score   support

           0     0.9144    0.8199    0.8645      6173
           1     0.7578    0.8801    0.8144      3954

    accuracy                         0.8434     10127
   macro avg     0.8361    0.8500    0.8395     10127
weighted avg     0.8532    0.8434    0.8450     10127



,threshold,precision,recall,f1,balanced_accuracy
0,0.450,0.757840,0.880121,0.814416,0.849991
1,0.475,0.768142,0.864694,0.813563,0.848757
2,0.375,0.731836,0.914517,0.813041,0.849936
3,0.425,0.746506,0.891502,0.812586,0.848797
4,0.400,0.738213,0.902883,0.812287,0.848898
5,0.500,0.777495,0.849267,0.811797,0.846794
6,0.350,0.723556,0.922104,0.810853,0.848222
7,0.325,0.714730,0.928933,0.807874,0.845723
8,0.525,0.784435,0.831057,0.807074,0.842387
9,0.300,0.705927,0.939808,0.806249,0.844519


## Step 7) Segment-Level Performance Check

Evaluate whether model quality is consistent across key business segments.
This helps detect overfitting to dominant groups and informs campaign fairness.

In [10]:
valid_view = X_valid.copy()
valid_view['y_true'] = y_valid.values
valid_view['y_pred'] = best_pred
valid_view['y_proba'] = best_proba

segment_cols = [c for c in ['age_group', 'income_group', 'offer_type'] if c in valid_view.columns]
seg_reports = {}
for col in segment_cols:
    seg_df = valid_view.groupby(col, observed=False).apply(
        lambda g: pd.Series({
            'records': len(g),
            'actual_rate': g['y_true'].mean(),
            'pred_rate': g['y_pred'].mean(),
            'precision': precision_score(g['y_true'], g['y_pred'], zero_division=0),
            'recall': recall_score(g['y_true'], g['y_pred'], zero_division=0)
        })
    ).reset_index()
    seg_df[['actual_rate', 'pred_rate', 'precision', 'recall']] = seg_df[['actual_rate', 'pred_rate', 'precision', 'recall']].round(4)
    seg_reports[col] = seg_df

for k, v in seg_reports.items():
    print('\nSegment report:', k)
    display(v.sort_values('records', ascending=False).head(15))


Segment report: age_group


,age_group,records,actual_rate,pred_rate,precision,recall
3,46-55,3170.0,0.3199,0.3644,0.7385,0.8412
5,66+,2311.0,0.4639,0.5530,0.7621,0.9086
4,56-65,2026.0,0.4477,0.5143,0.7841,0.9008
2,36-45,1213.0,0.4237,0.4913,0.7701,0.8930
1,26-35,769.0,0.3173,0.3862,0.6835,0.8320
0,18-25,638.0,0.3182,0.3511,0.7768,0.8571



Segment report: income_group


,income_group,records,actual_rate,pred_rate,precision,recall
3,middle,4023.0,0.3637,0.4077,0.7652,0.8578
2,lower_middle,2620.0,0.3859,0.4542,0.7597,0.8942
4,upper_middle,1456.0,0.5213,0.6250,0.7648,0.9170
1,low,1383.0,0.3030,0.3630,0.7032,0.8425
0,high,645.0,0.4682,0.5426,0.7771,0.9007


## Step 8) Export Pre-Model Artifacts

Save data splits, model scores, chosen threshold, and feature governance so later modeling notebooks remain reproducible.

In [11]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'premodel'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_export = X_train.copy()
train_export[target_col] = y_train.values
valid_export = X_valid.copy()
valid_export[target_col] = y_valid.values
valid_export['pred_proba_best'] = best_proba
valid_export['pred_label_best'] = best_pred
valid_export['svd_score'] = svd_scores.values
valid_export['popular_score'] = valid_proba_popular


train_path = OUTPUT_DIR / 'train_split.csv'
valid_path = OUTPUT_DIR / 'valid_split_with_preds.csv'
scores_path = OUTPUT_DIR / 'baseline_model_scores.csv'
threshold_path = OUTPUT_DIR / 'threshold_sweep.csv'
config_path = OUTPUT_DIR / 'premodel_config.json'

train_export.to_csv(train_path, index=False)
valid_export.to_csv(valid_path, index=False)

# Thêm 2 baselines mới vào results
results.append(popular_metrics)     # từ Step 4b
results.append(svd_metrics)         # từ Step 5b

# Rebuild results_df với tất cả 5 models
results_df = pd.DataFrame(results).sort_values('pr_auc', ascending=False).reset_index(drop=True)

results_df.to_csv(scores_path, index=False)
th_df.to_csv(threshold_path, index=False)

config = {
    'target_col': target_col,
    'best_model_name': best_model_name,
    'best_threshold': best_threshold,
    'dropped_id_like_cols': id_like_cols,
    'dropped_leakage_cols': leakage_cols,
    'n_numeric_features': len(numeric_cols),
    'n_categorical_features': len(categorical_cols),
    'n_total_features': len(feature_cols),
    'svd_k': k,                                    # ← THÊM
    'models_compared': list(results_df['model']),   # ← THÊM
}

with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

print('Saved artifacts:')
print('-', train_path)
print('-', valid_path)
print('-', scores_path)
print('-', threshold_path)
print('-', config_path)
results_df

Saved artifacts:
- c:\Nam\ddm\Data_marketing_starbuck_rewards\outputs\premodel\train_split.csv
- c:\Nam\ddm\Data_marketing_starbuck_rewards\outputs\premodel\valid_split_with_preds.csv
- c:\Nam\ddm\Data_marketing_starbuck_rewards\outputs\premodel\baseline_model_scores.csv
- c:\Nam\ddm\Data_marketing_starbuck_rewards\outputs\premodel\threshold_sweep.csv
- c:\Nam\ddm\Data_marketing_starbuck_rewards\outputs\premodel\premodel_config.json


,model,accuracy,balanced_accuracy,f1,precision,recall,roc_auc,pr_auc
0,hist_gb,0.846253,0.846794,0.811797,0.777495,0.849267,0.927503,0.872726
1,random_forest,0.824528,0.836702,0.798823,0.723099,0.892261,0.911085,0.832345
2,log_reg,0.807445,0.817962,0.778359,0.706854,0.865959,0.898018,0.807279
3,most_popular_offer,0.654587,0.604483,0.459351,0.590620,0.375822,0.669499,0.528805
4,svd_collaborative,0.609559,0.500000,0.000000,0.000000,0.000000,0.500000,0.390441
